### Clean and Restructure Dataset for training

In [3]:
import os
import shutil
import sys

# Add my_libs to Python path
sys.path.insert(0, '/home/i_am_helium/my_libs')
import pandas as pd

BASE_PATH = "skin"
OUTPUT_PATH = "skin_clean_R_G_IR"

classes = ["bcc", "mel", "nevus", "bkl"]

manifest_rows = []
excluded_rows = []


def find_channel_files(lesion_path):
    """
    Return paths of R, G, IR images inside lesion_path/in.
    """
    in_path = os.path.join(lesion_path, "in")

    if not os.path.exists(in_path):
        return None, None, None, "no /in/ folder"

    files = os.listdir(in_path)

    r_files = [
        f for f in files
        if f.startswith("R") and not f.startswith("IR")
    ]

    g_files = [
        f for f in files
        if f.startswith("G")
    ]

    ir_files = [
        f for f in files
        if f.startswith("IR")
    ]

    missing = []
    if len(r_files) == 0:
        missing.append("R")
    if len(g_files) == 0:
        missing.append("G")
    if len(ir_files) == 0:
        missing.append("IR")

    if missing:
        return None, None, None, f"lack: {', '.join(missing)}"

    r_path = os.path.join(in_path, r_files[0])
    g_path = os.path.join(in_path, g_files[0])
    ir_path = os.path.join(in_path, ir_files[0])

    return r_path, g_path, ir_path, None


def copy_valid_lesion(cls_name, lesion_name, lesion_path):
    r_path, g_path, ir_path, reason = find_channel_files(lesion_path)

    if reason is not None:
        excluded_rows.append({
            "class": cls_name,
            "lesion_name": lesion_name,
            "original_path": lesion_path,
            "reason": reason
        })
        return False

    output_lesion_dir = os.path.join(OUTPUT_PATH, cls_name, lesion_name)
    os.makedirs(output_lesion_dir, exist_ok=True)

    # Copy and rename files to standard names
    r_out = os.path.join(output_lesion_dir, "R.png")
    g_out = os.path.join(output_lesion_dir, "G.png")
    ir_out = os.path.join(output_lesion_dir, "IR.png")

    shutil.copy2(r_path, r_out)
    shutil.copy2(g_path, g_out)
    shutil.copy2(ir_path, ir_out)

    manifest_rows.append({
        "class": cls_name,
        "lesion_name": lesion_name,
        "R_path": r_out,
        "G_path": g_out,
        "IR_path": ir_out,
        "original_path": lesion_path
    })

    return True


# Create output folder
os.makedirs(OUTPUT_PATH, exist_ok=True)

for cls in classes:
    cls_path = os.path.join(BASE_PATH, cls)

    if cls != "bkl":
        lesions = [
            f for f in os.listdir(cls_path)
            if os.path.isdir(os.path.join(cls_path, f))
        ]

        for lesion in lesions:
            lesion_path = os.path.join(cls_path, lesion)
            copy_valid_lesion(cls, lesion, lesion_path)

    else:
        # bkl has intermediate folders such as L82, L81.4, etc.
        intermediate_folders = [
            f for f in os.listdir(cls_path)
            if os.path.isdir(os.path.join(cls_path, f))
        ]

        for inter_folder in intermediate_folders:
            inter_path = os.path.join(cls_path, inter_folder)

            lesions = [
                f for f in os.listdir(inter_path)
                if os.path.isdir(os.path.join(inter_path, f))
            ]

            for lesion in lesions:
                lesion_path = os.path.join(inter_path, lesion)

                # Keep intermediate folder name to avoid duplicate lesion names
                clean_lesion_name = f"{inter_folder}__{lesion}"

                copy_valid_lesion(cls, clean_lesion_name, lesion_path)


# Save CSV files
manifest_df = pd.DataFrame(manifest_rows)
excluded_df = pd.DataFrame(excluded_rows)

manifest_csv = os.path.join(OUTPUT_PATH, "manifest.csv")
excluded_csv = os.path.join(OUTPUT_PATH, "excluded_lesions.csv")

manifest_df.to_csv(manifest_csv, index=False)
excluded_df.to_csv(excluded_csv, index=False)

print("Clean dataset created successfully!")
print(f"Output folder: {OUTPUT_PATH}")
print(f"Valid lesions: {len(manifest_df)}")
print(f"Excluded lesions: {len(excluded_df)}")

print("\nClass distribution:")
print(manifest_df["class"].value_counts())

print(f"\nManifest saved to: {manifest_csv}")
print(f"Excluded lesions saved to: {excluded_csv}")

Clean dataset created successfully!
Output folder: skin_clean_R_G_IR
Valid lesions: 1765
Excluded lesions: 9

Class distribution:
class
mel      538
nevus    527
bcc      450
bkl      250
Name: count, dtype: int64

Manifest saved to: skin_clean_R_G_IR/manifest.csv
Excluded lesions saved to: skin_clean_R_G_IR/excluded_lesions.csv


### Try with Hair Preprocessing

In [2]:
import sys
print(sys.executable)

/opt/jupyterhub/bin/python3


In [3]:
!which python

/opt/jupyterhub/bin/python


In [4]:
!pip show opencv-python-headless

Name: opencv-python-headless
Version: 4.13.0.92
Summary: Wrapper package for OpenCV python bindings.
Home-page: https://github.com/opencv/opencv-python
Author: 
Author-email: 
License: Apache 2.0
Location: /home/i_am_helium/.local/lib/python3.10/site-packages
Requires: numpy
Required-by: 


In [7]:
import sys
!{sys.executable} -m pip install opencv-python-headless --target=/home/i_am_helium/my_libs

  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (60.4 MB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)


In [8]:
import sys
sys.path.insert(0, "/home/i_am_helium/my_libs")

import cv2
print("cv2 version:", cv2.__version__)

cv2 version: 4.13.0
